In [8]:
import sys

import os

import glob

import json

import importlib.util

import re

import warnings

import numpy as np

import pandas as pd

import networkx as nx

import matplotlib

import matplotlib.pyplot as plt

import torch

import torch.nn as nn

import torch.nn.functional as F

from torch_geometric.data import Data

from torch_geometric.nn import GATv2Conv

from concurrent.futures import ProcessPoolExecutor, as_completed

from collections import defaultdict

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

##### ==========================================#####

##### WORKER CODE (for parallel processing)     #####

##### ==========================================#####

WORKER_CODE = """

import re

import itertools

import numpy as np

import pandas as pd

from scipy.stats import entropy

import warnings

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

SAMPLE_SIZE = 500

SIGNATURE_SIZE = 200

UUID_RE = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$', re.I)

def normalize_scalar(v):

    if pd.isna(v):

        return ''

    s = str(v).strip().casefold()

    # Standardize obvious numeric codes while preserving non-numeric ids.

    if re.fullmatch(r'[-+]?\d+(\.0+)?', s):

        s = s.split('.')[0]

        s = s.lstrip('+')

        s = s.lstrip('0') or '0'

    return s

def smart_normalize(series):

    s = series.map(normalize_scalar)

    try:

        nums = pd.to_numeric(s, errors='coerce')

        num_mask = nums.notna()

        if num_mask.mean() > 0.8:

            s.loc[num_mask] = nums.loc[num_mask].astype(float).round(8).astype(str)

    except Exception:

        pass

    try:

        dts = pd.to_datetime(s, errors='coerce', utc=True)

        dt_mask = dts.notna()

        if dt_mask.mean() > 0.8:

            s.loc[dt_mask] = dts.loc[dt_mask].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

    except Exception:

        pass

    return s

def get_id_score(col_name):

    col = col_name.lower()

    if col in ['id', 'pk', 'key', 'uuid', 'guid', 'index', '_id']:

        return 3

    if re.search(r'(_id|id|_key|key|_pk|pk)$', col):

        return 2

    if re.search(r'(id|key|code|num|no|uuid|guid)', col):

        return 1

    return 0

def get_signature(series, n=200):

    uniques = series.dropna().unique()

    if len(uniques) > n:

        try:

            uniques = np.random.choice(uniques, n, replace=False)

        except Exception:

            pass

    return set(hash(str(x)) for x in uniques)

def profile_atomic_values(vals, col_name):

    n = max(1, len(vals))

    counts = vals.value_counts(dropna=False)

    numeric_parseable = pd.to_numeric(vals, errors='coerce').notna().mean() if n else 0.0

    date_parseable = pd.to_datetime(vals, errors='coerce', utc=True).notna().mean() if n else 0.0

    avg_len = vals.str.len().mean() if n else 0.0

    uuid_like_name = 1.0 if re.search(r'(uuid|guid)', col_name.lower()) else 0.0

    uuid_valid_ratio = vals.map(lambda x: bool(UUID_RE.match(str(x)))).mean() if n else 0.0

    text_share = max(0.0, 1.0 - max(numeric_parseable, date_parseable))

    type_consistency = max(numeric_parseable, date_parseable, text_share)

    major_is_num = 1.0 if numeric_parseable >= max(date_parseable, text_share) else 0.0

    major_is_date = 1.0 if date_parseable >= max(numeric_parseable, text_share) else 0.0

    major_is_text = 1.0 if text_share >= max(numeric_parseable, date_parseable) else 0.0

    top_freq = []

    for k, v in counts.head(3).items():

        top_freq.append(f"{str(k)[:20]}:{int(v)}")

    return {

        'numeric_parseable': float(numeric_parseable),

        'date_parseable': float(date_parseable),

        'avg_len': float(avg_len),

        'uuid_like_name': float(uuid_like_name),

        'uuid_valid_ratio': float(uuid_valid_ratio),

        'invalid_uuid_rate': float(1.0 - uuid_valid_ratio) if uuid_like_name > 0 else 0.0,

        'type_consistency': float(type_consistency),

        'major_is_num': float(major_is_num),

        'major_is_date': float(major_is_date),

        'major_is_text': float(major_is_text),

        'top_freq': ' | '.join(top_freq)

    }

def extract_features(df, cols):

    is_composite = isinstance(cols, tuple)

    col_names = list(cols) if is_composite else [cols]

    n_rows = len(df)

    limit = SAMPLE_SIZE

    try:

        head = df[col_names].head(limit)

        tail = df[col_names].iloc[limit:limit * 2] if n_rows > limit * 2 else df[col_names].tail(limit)

        rand = df[col_names].sample(min(n_rows, limit), random_state=42) if n_rows > limit else df[col_names]

    except Exception:

        return None

    if is_composite:

        fn = lambda x: '||'.join(normalize_scalar(v) for v in x)

        vals_head = head.agg(fn, axis=1)

        vals_tail = tail.agg(fn, axis=1)

        vals_rand = rand.agg(fn, axis=1)

        name_str = "_".join(cols).lower()

        col_count = len(cols)

        id_score = max(get_id_score(c) for c in cols)

        profile = {

            'numeric_parseable': 0.0,

            'date_parseable': 0.0,

            'avg_len': float(vals_rand.str.len().mean()) if len(vals_rand) > 0 else 0.0,

            'uuid_like_name': 0.0,

            'uuid_valid_ratio': 0.0,

            'invalid_uuid_rate': 0.0,

            'type_consistency': 1.0,

            'major_is_num': 0.0,

            'major_is_date': 0.0,

            'major_is_text': 1.0,

            'top_freq': ''

        }

    else:

        vals_head = smart_normalize(head[cols])

        vals_tail = smart_normalize(tail[cols])

        vals_rand = smart_normalize(rand[cols])

        name_str = cols.lower()

        col_count = 1

        id_score = get_id_score(cols)

        profile = profile_atomic_values(vals_rand, cols)

    # Robustness check over head/random/tail samples.

    u_scores = []

    for v in [vals_head, vals_rand, vals_tail]:

        u_scores.append(v.nunique() / len(v) if len(v) > 0 else 0)

    if (max(u_scores) - min(u_scores)) > 0.8:

        return None

    vals = vals_rand

    n_uniq = vals.nunique()

    uniqueness = sorted(u_scores)[1]

    null_rate = df[col_names].isna().mean().max()

    dup_rate = 1.0 - uniqueness

    card_log = np.log1p(n_uniq)

    counts = vals.value_counts()

    ent = entropy(counts) if len(counts) > 0 else 0

    is_uuid_name = 1.0 if re.search(r'(uuid|guid)', name_str) else 0.0

    is_time_name = 1.0 if re.search(r'(date|time|created)', name_str) else 0.0

    # 15-D feature vector.

    feats = [

        float(uniqueness),

        float(null_rate),

        float(dup_rate),

        float(card_log),

        float(ent),

        float(profile['numeric_parseable']),

        float(profile['avg_len']),

        float(id_score),

        float(is_uuid_name),

        float(is_time_name),

        float(col_count),

        float(profile['major_is_num']),

        float(profile['major_is_date']),

        float(profile['major_is_text']),

        float(profile['type_consistency'])

    ]

    sig = get_signature(vals, SIGNATURE_SIZE)

    return feats, sig, float(profile['avg_len']), n_rows, profile, float(n_uniq)

def minimal_composite_prune(composites):

    # Drop supersets when a subset has near-identical uniqueness.

    if not composites:

        return []

    composites = sorted(composites, key=lambda x: (len(x['cols']), -x['features'][0]))

    keep = []

    for cand in composites:

        cols_set = set(cand['cols'])

        is_superset = False

        for chosen in keep:

            chosen_set = set(chosen['cols'])

            if chosen_set.issubset(cols_set):

                if abs(chosen['features'][0] - cand['features'][0]) <= 0.01:

                    is_superset = True

                    break

        if not is_superset:

            keep.append(cand)

    return keep

def process_table_file(filepath):

    try:

        try:

            df = pd.read_csv(filepath, on_bad_lines='skip', engine='pyarrow')

        except Exception:

            try:

                df = pd.read_csv(filepath, on_bad_lines='skip', low_memory=False)

            except Exception:

                df = pd.read_csv(filepath, on_bad_lines='skip', encoding='latin-1', low_memory=False)

        if df.empty:

            return []

        cols = [c for c in df.columns if not re.search(r'^(desc|note|comment|text)', c, re.I)]

        table = filepath.split('/')[-1].replace('.csv', '')

        candidates = []

        atomic_pool = []

        # 1) Atomic candidates.

        for col in cols:

            res = extract_features(df, col)

            if res:

                feats, sig, avg_len, n_rows, profile, cardinality = res

                row = {

                    'table': table,

                    'cols': (col,),

                    'features': feats,

                    'signature': sig,

                    'avg_len': avg_len,

                    'n_rows': n_rows,

                    'profile': profile,

                    'cardinality': cardinality

                }

                candidates.append(row)

                atomic_pool.append(row)

        # 2) Composite candidates from top-m by uniqueness/entropy (even if singles are weak).

        atomic_ranked = sorted(

            atomic_pool,

            key=lambda x: (x['features'][0], x['features'][4], x['features'][7]),

            reverse=True

        )

        top_m = [x['cols'][0] for x in atomic_ranked[:min(12, len(atomic_ranked))]]

        composites = []

        for r in range(2, 4):

            for c_cols in itertools.combinations(top_m, r):

                res = extract_features(df, c_cols)

                if res:

                    feats, sig, avg_len, n_rows, profile, cardinality = res

                    if feats[0] > 0.80:

                        composites.append({

                            'table': table,

                            'cols': c_cols,

                            'features': feats,

                            'signature': sig,

                            'avg_len': avg_len,

                            'n_rows': n_rows,

                            'profile': profile,

                            'cardinality': cardinality

                        })

        candidates.extend(minimal_composite_prune(composites))

        return candidates

    except Exception as e:

        print(f"  WARN: Failed to process {filepath}: {e}")

        return []

"""

CONFIG = {

    'folder': './data',

    'device': 'cpu',

    'tau': 0.25,

    'u_min': 0.99,

    'k_max': 50,

    'min_overlap': 2,

    'num_workers': max(1, (os.cpu_count() or 4) - 1),

    'hidden_dim': 64,

    'heads': 4,

    'dropout': 0.1,

    'lr': 0.005,

    'weight_decay': 1e-4,

    'epochs': 150,

    'use_fk_edge_model': True,

    'fk_edge_epochs': 120,

    'fk_edge_prob': 0.55,

    'fk_pos_name_min': 0.5,

    'fk_pos_inc_min': 0.6,

    'fk_neg_inc_max': 0.4,

    'fk_neg_name_max': 1.0,

    'auto_tune': False,

    'manual_pk_labels': [],

    'manual_fk_labels': [],
    'acceptance_min_acc': 0.90,
    'acceptance_min_precision': 0.85,
    'acceptance_min_recall': 0.90,
    'acceptance_min_f1': 0.90,
    'acceptance_max_train_val_f1_gap': 0.08

}

##### ==========================================#####

##### HELPER FUNCTIONS                          #####

##### ==========================================#####

UUID_RE = re.compile(r'^[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$', re.I)

def clean_table_name(name):

    name = re.sub(r'^asm_', '', name)

    name = re.sub(r'_\d{12,}$', '', name)

    return name.strip('_')

def normalize_scalar(v):

    if pd.isna(v):

        return ''

    s = str(v).strip().casefold()

    if re.fullmatch(r'[-+]?\d+(\.0+)?', s):

        s = s.split('.')[0]

        s = s.lstrip('+')

        s = s.lstrip('0') or '0'

    return s

def normalize_series(series):

    s = series.map(normalize_scalar)

    try:

        nums = pd.to_numeric(s, errors='coerce')

        num_mask = nums.notna()

        if num_mask.mean() > 0.8:

            s.loc[num_mask] = nums.loc[num_mask].astype(float).round(8).astype(str)

    except Exception:

        pass

    try:

        dts = pd.to_datetime(s, errors='coerce', utc=True)

        dt_mask = dts.notna()

        if dt_mask.mean() > 0.8:

            s.loc[dt_mask] = dts.loc[dt_mask].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

    except Exception:

        pass

    return s

def get_name_similarity(col_name, table_name):

    col = col_name.lower()

    tbl = clean_table_name(table_name).lower()

    col_base = re.sub(r'_?(id|key|code|no|num)$', '', col)

    if col_base and col_base == tbl:

        return 1.0

    if col_base and (col_base + 's' == tbl or col_base + 'es' == tbl):

        return 0.9

    if col_base and (tbl + 's' == col_base or tbl + 'es' == col_base):

        return 0.9

    if col_base and len(tbl) >= 3 and tbl in col_base:

        return 0.7

    col_nopfx = re.sub(r'^[a-z]{1,2}_', '', col_base)

    if col_nopfx and col_nopfx != col_base:

        if col_nopfx == tbl:

            return 1.0

        if col_nopfx + 's' == tbl or col_nopfx + 'es' == tbl:

            return 0.9

        if tbl + 's' == col_nopfx or tbl + 'es' == col_nopfx:

            return 0.9

        if len(tbl) >= 3 and tbl in col_nopfx:

            return 0.7

        if len(col_nopfx) >= 3 and col_nopfx in tbl:

            return 0.7

    return 0.0

def load_csv_column(filepath, col_name):

    try:

        try:

            df = pd.read_csv(filepath, usecols=[col_name], on_bad_lines='skip', engine='pyarrow')

        except Exception:

            df = pd.read_csv(filepath, usecols=[col_name], on_bad_lines='skip', low_memory=False)

        return normalize_series(df[col_name].dropna())

    except Exception:

        return pd.Series(dtype=str)

##### ==========================================#####

##### FK INFERENCE                              #####

##### ==========================================#####

def infer_foreign_keys(candidates, schema, data, config):

    print("\n  Phase 5a: Inferring Foreign Keys...")

    def train_fk_edge_model(rows, cfg):

        labels = np.array([r[1] for r in rows], dtype=np.float32)

        pos = int(labels.sum())

        neg = int(len(labels) - pos)

        if len(rows) < 2 or pos == 0 or neg == 0:

            return None, None

        idx = np.arange(len(rows))

        np.random.shuffle(idx)

        split = max(1, int(0.8 * len(idx)))

        train_idx, val_idx = idx[:split], idx[split:]

        if len(val_idx) == 0:

            return None, None

        X = torch.tensor([rows[i][0] for i in range(len(rows))], dtype=torch.float)

        y = torch.tensor([rows[i][1] for i in range(len(rows))], dtype=torch.float)

        train_mask = torch.zeros(len(rows), dtype=torch.bool)

        val_mask = torch.zeros(len(rows), dtype=torch.bool)

        train_mask[train_idx] = True

        val_mask[val_idx] = True

        model = nn.Sequential(

            nn.Linear(8, 32),

            nn.ReLU(),

            nn.Dropout(cfg.get('dropout', 0.1)),

            nn.Linear(32, 1)

        )

        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.get('lr', 0.005), weight_decay=cfg.get('weight_decay', 1e-4))

        pos_weight = torch.tensor([max(1.0, neg / max(pos, 1))], dtype=torch.float)

        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        best_f1 = -1.0

        best_state = None

        edge_epochs = int(cfg.get('fk_edge_epochs', 120))

        for _ in range(edge_epochs):

            model.train()

            optimizer.zero_grad()

            logits = model(X).squeeze()

            loss = criterion(logits[train_mask], y[train_mask])

            loss.backward()

            optimizer.step()

            model.eval()

            with torch.no_grad():

                probs = torch.sigmoid(model(X).squeeze())

                preds = (probs[val_mask] > 0.5).float().cpu().numpy()

                y_val = y[val_mask].cpu().numpy()

                f1 = f1_score(y_val, preds, zero_division=0)

                if f1 > best_f1:

                    best_f1 = f1

                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

        if best_state is None:

            return None, None

        model.load_state_dict(best_state)

        return model, {'best_val_f1': best_f1, 'train_n': int(train_mask.sum()), 'val_n': int(val_mask.sum())}

    pk_map = {}

    for table, entries in schema['tables'].items():

        for entry in entries:

            if entry['type'] == 'PK':

                pk_map[table] = entry['columns']

    atomic_by_table = defaultdict(dict)

    for i, c in enumerate(candidates):

        if len(c['cols']) == 1:

            atomic_by_table[c['table']][c['cols'][0]] = i

    fk_pairs = {}

    for pk_table, pk_cols in pk_map.items():

        for pk_col in pk_cols:

            for other_table, col_map in atomic_by_table.items():

                if other_table == pk_table:

                    continue

                for other_col, cand_idx in col_map.items():

                    c = candidates[cand_idx]

                    other_pk = pk_map.get(other_table, [])

                    if len(other_pk) == 1 and other_pk[0] == other_col:

                        continue

                    name_sim = get_name_similarity(other_col, pk_table)

                    col_match = (other_col.lower() == pk_col.lower())

                    if name_sim >= 0.7 or col_match:

                        if c['features'][7] > 0:

                            key = (other_table, other_col, pk_table, pk_col)

                            fk_pairs[key] = {

                                'name_sim': name_sim,

                                'inc_score': 0.0,

                                'src_id_score': c['features'][7],

                            }

    edge_index = data.edge_index.cpu().numpy()

    edge_attr = data.edge_attr.cpu().numpy()

    edge_records = []

    for k in range(edge_index.shape[1]):

        src, dst = edge_index[0, k], edge_index[1, k]

        c_src, c_dst = candidates[src], candidates[dst]

        if c_src['table'] == c_dst['table']:

            continue

        if len(c_src['cols']) != 1 or len(c_dst['cols']) != 1:

            continue

        dst_table = c_dst['table']

        dst_col = c_dst['cols'][0]

        src_table = c_src['table']

        src_col = c_src['cols'][0]

        if dst_table not in pk_map or dst_col not in pk_map[dst_table]:

            continue

        if c_src['features'][7] == 0:

            continue

        src_pk = pk_map.get(src_table, [])

        if len(src_pk) == 1 and src_pk[0] == src_col:

            continue

        inc_a_b = float(edge_attr[k][0])

        inc_b_a = float(edge_attr[k][1])

        jacc = float(edge_attr[k][2])

        contain = float(edge_attr[k][3])

        weighted = float(edge_attr[k][4])

        card_ratio = float(edge_attr[k][5])

        type_compat = float(edge_attr[k][6])

        len_compat = float(edge_attr[k][7])

        name_sim = float(get_name_similarity(src_col, dst_table))

        src_id_score = float(c_src['features'][7])

        feat8 = [inc_a_b, inc_b_a, jacc, contain, weighted, card_ratio, type_compat, len_compat]

        weak_label = None

        pos_name_min = float(config.get('fk_pos_name_min', 0.5))

        pos_inc_min = float(config.get('fk_pos_inc_min', 0.6))

        if name_sim >= pos_name_min and inc_a_b >= pos_inc_min:

            weak_label = 1.0

        elif inc_a_b < pos_inc_min:

            weak_label = 0.0

        edge_records.append({

            'key': (src_table, src_col, dst_table, dst_col),

            'name_sim': name_sim,

            'inc_score': inc_a_b,

            'src_id_score': src_id_score,

            'feat8': feat8,

            'weak_label': weak_label,

            'inc_b_a': inc_b_a,

            'jaccard': jacc,

            'containment': contain,

            'weighted_overlap': weighted,

            'card_ratio': card_ratio,

            'type_compat': type_compat,

            'len_compat': len_compat

        })

    if len(edge_records) == 0 and len(fk_pairs) > 0:

        for key, info in fk_pairs.items():

            src_table, src_col, dst_table, dst_col = key

            name_sim = float(info.get('name_sim', 0.0))

            inc_score = float(info.get('inc_score', 0.0))

            src_id_score = float(info.get('src_id_score', 0.0))

            feat8 = [inc_score, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

            weak_label = 1.0 if name_sim >= 0.7 else 0.0

            edge_records.append({

                'key': key,

                'name_sim': name_sim,

                'inc_score': inc_score,

                'src_id_score': src_id_score,

                'feat8': feat8,

                'weak_label': weak_label,

                'inc_b_a': 0.0,

                'jaccard': 0.0,

                'containment': 0.0,

                'weighted_overlap': 0.0,

                'card_ratio': 0.0,

                'type_compat': 0.0,

                'len_compat': 0.0

            })

    for rec in edge_records:

        key = rec['key']

        if key in fk_pairs:

            fk_pairs[key]['inc_score'] = max(fk_pairs[key]['inc_score'], rec['inc_score'])

        elif rec['name_sim'] >= 0.5:

            fk_pairs[key] = {

                'name_sim': rec['name_sim'],

                'inc_score': rec['inc_score'],

                'src_id_score': rec['src_id_score'],

            }

    fk_edge_model = None

    if config.get('use_fk_edge_model', True):

        labeled_rows = [(r['feat8'], r['weak_label']) for r in edge_records if r['weak_label'] is not None]

        for src_t, src_c, dst_t, dst_c, y in config.get('manual_fk_labels', []):

            for r in edge_records:

                s_t, s_c, d_t, d_c = r['key']

                if s_t == src_t and s_c == src_c and d_t == dst_t and d_c == dst_c:

                    labeled_rows.append((r['feat8'], float(y)))

                    break

        n_pos = sum(1 for _, y in labeled_rows if y == 1.0)

        n_neg = sum(1 for _, y in labeled_rows if y == 0.0)

        print(f"    > FK weak labels: pos={n_pos}, neg={n_neg}, total={len(labeled_rows)}")

        if n_pos == 0 or n_neg == 0:

            unlabeled = [r for r in edge_records if r['weak_label'] is None]

            unlabeled = sorted(unlabeled, key=lambda r: r['inc_score'])

            if len(unlabeled) > 0:

                k_backfill = max(1, min(8, len(unlabeled) // 4 if len(unlabeled) >= 4 else len(unlabeled)))

                if n_neg == 0:

                    for r in unlabeled[:k_backfill]:

                        labeled_rows.append((r['feat8'], 0.0))

                if n_pos == 0:

                    for r in unlabeled[-k_backfill:]:

                        labeled_rows.append((r['feat8'], 1.0))

        n_pos = sum(1 for _, y in labeled_rows if y == 1.0)

        n_neg = sum(1 for _, y in labeled_rows if y == 0.0)

        if len(labeled_rows) > 0 and (n_pos == 0 or n_neg == 0):

            seed_feats = [feat for feat, _ in labeled_rows[:min(6, len(labeled_rows))]]

            if n_neg == 0:

                for feat in seed_feats:

                    labeled_rows.append((feat, 0.0))

            if n_pos == 0:

                for feat in seed_feats:

                    labeled_rows.append((feat, 1.0))

        fk_edge_model, fk_edge_info = train_fk_edge_model(labeled_rows, config)

        if fk_edge_model is not None:

            print(f"    > FK edge model trained (train={fk_edge_info['train_n']}, val={fk_edge_info['val_n']}, best val F1={fk_edge_info['best_val_f1']:.3f})")

            fk_prob_thr = float(config.get('fk_edge_prob', 0.75))

            fk_edge_model.eval()

            with torch.no_grad():

                for rec in edge_records:

                    key = rec['key']

                    if key in fk_pairs:

                        continue

                    if rec['name_sim'] >= 0.5:

                        continue

                    x = torch.tensor(rec['feat8'], dtype=torch.float).unsqueeze(0)

                    p = float(torch.sigmoid(fk_edge_model(x)).squeeze().item())

                    if p >= fk_prob_thr:

                        fk_pairs[key] = {

                            'name_sim': rec['name_sim'],

                            'inc_score': rec['inc_score'],

                            'src_id_score': rec['src_id_score'],

                            'edge_prob': p

                        }

        else:

            print("    > FK edge model skipped (insufficient balanced weak labels)")

    print(f"    > {len(fk_pairs)} FK candidates from strategy cascade")

    data_folder = config['folder']

    col_cache = {}

    def get_col_values(table, col):

        k = (table, col)

        if k not in col_cache:

            filepath = os.path.join(data_folder, f"{table}.csv")

            col_cache[k] = set(load_csv_column(filepath, col).unique())

        return col_cache[k]

    validated = []

    for (src_table, src_col, dst_table, dst_col), info in fk_pairs.items():

        try:

            fk_vals = get_col_values(src_table, src_col)

            pk_vals = get_col_values(dst_table, dst_col)

        except Exception:

            continue

        if not fk_vals or not pk_vals:

            continue

        inter = fk_vals & pk_vals

        inc_a_b = len(inter) / max(1, len(fk_vals))

        inc_b_a = len(inter) / max(1, len(pk_vals))

        jacc = len(inter) / max(1, len(fk_vals | pk_vals))

        card_ok = len(fk_vals) <= len(pk_vals) * 1.10

        if inc_a_b >= 0.95 and card_ok:

            edge_signal = float(info.get('edge_prob', info.get('inc_score', 0.0)))

            confidence = inc_a_b * 0.55 + info['name_sim'] * 0.20 + min(edge_signal, 1.0) * 0.15 + jacc * 0.10

            reason = f"inc(A->B)={inc_a_b:.2f}, inc(B->A)={inc_b_a:.2f}, jacc={jacc:.2f}, name={info['name_sim']:.2f}"

            validated.append({

                'Source Table': src_table,

                'Source Col': [src_col],

                'Target Table': dst_table,

                'Target Col': dst_col,

                'Confidence': f"{confidence:.2f}",

                'Inclusion_A_to_B': f"{inc_a_b:.2f}",

                'Inclusion_B_to_A': f"{inc_b_a:.2f}",

                'Jaccard': f"{jacc:.2f}",

                'Reason': reason

            })

    best_fk = {}

    for v in validated:

        key = (v['Source Table'], tuple(v['Source Col']))

        if key not in best_fk or float(v['Confidence']) > float(best_fk[key]['Confidence']):

            best_fk[key] = v

    validated = sorted(best_fk.values(), key=lambda x: -float(x['Confidence']))

    print(f"    > {len(validated)} FK relationships validated with data")

    return validated

##### ==========================================#####

##### REFINEMENT & EXPORT                       #####

##### ==========================================#####

def refine_and_export(candidates, schema, data, config):

    print("\nPhase 5: Refinement & Exporting...")

    relationships = infer_foreign_keys(candidates, schema, data, config)

    inbound_refs = defaultdict(int)

    for r in relationships:

        inbound_refs[r['Target Table']] += 1

    centrality = {}

    if relationships:

        g = nx.DiGraph()

        for r in relationships:

            g.add_edge(r['Source Table'], r['Target Table'])

        try:

            centrality = nx.pagerank(g)

        except Exception:

            centrality = {n: 0.0 for n in g.nodes()}

    for r in relationships:

        table = r['Source Table']

        if table not in schema['tables']:

            schema['tables'][table] = []

        schema['tables'][table].append({

            'type': 'FK',

            'columns': r['Source Col'],

            'references': {'table': r['Target Table'], 'column': r['Target Col']},

            'explainability': {

                'confidence': r['Confidence'],

                'inclusion_a_to_b': r['Inclusion_A_to_B'],

                'inclusion_b_to_a': r['Inclusion_B_to_A'],

                'jaccard': r['Jaccard'],

                'reason': r['Reason'],

                'target_inbound_refs': inbound_refs[r['Target Table']],

                'target_centrality': round(float(centrality.get(r['Target Table'], 0.0)), 6)

            }

        })

    if relationships:

        pd.DataFrame(relationships).to_csv('relationships.csv', index=False)

        print(f"  > Saved: relationships.csv ({len(relationships)} relations found)")

    else:

        print('  > No strong FK relationships detected.')

    try:

        if not relationships:

            print('  > Skipping ERD (No relationships to draw).')

            return schema

        print('  > Drawing ERD (Connected Tables Only)...')

        G_viz = nx.DiGraph()

        for r in relationships:

            src_label = clean_table_name(r['Source Table'])

            dst_label = clean_table_name(r['Target Table'])

            G_viz.add_edge(src_label, dst_label, label=f"{r['Source Col'][0]}->{r['Target Col']}")

        plt.figure(figsize=(16, 12))

        pos = nx.spring_layout(G_viz, k=0.3, iterations=50, seed=42)

        nx.draw_networkx_nodes(G_viz, pos, node_size=2500, node_color='#E0F7FA', edgecolors='#006064')

        nx.draw_networkx_labels(G_viz, pos, font_size=8, font_weight='bold')

        nx.draw_networkx_edges(G_viz, pos, edge_color='#455A64', arrowstyle='-|>', arrowsize=15, width=1.2)

        edge_labels = nx.get_edge_attributes(G_viz, 'label')

        nx.draw_networkx_edge_labels(G_viz, pos, edge_labels, font_size=6)

        plt.title(f"Inferred Schema ({len(relationships)} Relations)", fontsize=16)

        plt.axis('off')

        plt.tight_layout()

        plt.savefig('erd.png', dpi=300)

        plt.close()

        print('  > Saved: erd.png')

    except Exception as e:

        print(f"  > Error drawing ERD: {e}")

    return schema

##### ==========================================#####

##### GNN MODEL & GRAPH                         #####

##### ==========================================#####

class SieveGNN(nn.Module):

    def __init__(self, in_dim, hidden_dim, heads, dropout, edge_dim):

        super().__init__()

        self.conv1 = GATv2Conv(in_dim, hidden_dim, heads=heads, edge_dim=edge_dim, dropout=dropout)

        self.conv2 = GATv2Conv(hidden_dim * heads, 2, heads=1, edge_dim=edge_dim, dropout=dropout)

    def forward(self, data):

        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        x = F.elu(self.conv1(x, edge_index, edge_attr))

        x = self.conv2(x, edge_index, edge_attr)

        return F.log_softmax(x, dim=1)

def type_compat_score(a, b):

    a_flags = a['features'][11:14]

    b_flags = b['features'][11:14]

    return float(np.dot(a_flags, b_flags))

def build_graph_fast(candidates):

    print('Phase 2: Building Graph...')

    # Blocking by cardinality bins + dominant type flags to avoid O(N^2).

    blocks = defaultdict(list)

    for idx, c in enumerate(candidates):

        card_bin = int(min(20, c.get('cardinality', 0) // 1000))

        type_bin = int(np.argmax(c['features'][11:14]))

        blocks[(card_bin, type_bin)].append(idx)

    index = defaultdict(list)

    for _, idx_list in blocks.items():

        for idx in idx_list:

            sig = candidates[idx].get('signature', set())

            if not sig:

                continue

            for token in sig:

                index[(token,)].append(idx)

    pair_counts = defaultdict(int)

    for _, indices in index.items():

        if len(indices) < 2 or len(indices) > CONFIG['k_max']:

            continue

        for i in range(len(indices)):

            for j in range(i + 1, len(indices)):

                idx_a, idx_b = indices[i], indices[j]

                if candidates[idx_a]['table'] == candidates[idx_b]['table']:

                    continue

                pair = tuple(sorted((idx_a, idx_b)))

                pair_counts[pair] += 1

    edge_index, edge_attr = [], []

    for (i, j), overlap in pair_counts.items():

        if overlap < CONFIG['min_overlap']:

            continue

        c_i, c_j = candidates[i], candidates[j]

        len_i, len_j = len(c_i['signature']), len(c_j['signature'])

        if len_i == 0 or len_j == 0:

            continue

        inter = overlap

        inc_i_j = inter / len_i

        inc_j_i = inter / len_j

        jacc = inter / max(1, (len_i + len_j - inter))

        containment = inter / max(1, min(len_i, len_j))

        weighted = inter / max(1, np.sqrt(len_i * len_j))

        card_ratio = min(c_i.get('cardinality', 1.0), c_j.get('cardinality', 1.0)) / max(c_i.get('cardinality', 1.0), c_j.get('cardinality', 1.0), 1.0)

        type_compat = type_compat_score(c_i, c_j)

        len_compat = min(c_i['avg_len'], c_j['avg_len']) / max(c_i['avg_len'], c_j['avg_len'], 0.01)

        if max(inc_i_j, inc_j_i) > CONFIG['tau']:

            e_ij = [inc_i_j, inc_j_i, jacc, containment, weighted, card_ratio, type_compat, len_compat]

            e_ji = [inc_j_i, inc_i_j, jacc, containment, weighted, card_ratio, type_compat, len_compat]

            edge_index.append([i, j])

            edge_attr.append(e_ij)

            edge_index.append([j, i])

            edge_attr.append(e_ji)

    x = torch.tensor([c['features'] for c in candidates], dtype=torch.float).nan_to_num()

    if not edge_index:

        edge_index = torch.tensor([[0], [0]], dtype=torch.long)

        edge_attr = torch.zeros((1, 8), dtype=torch.float)

    else:

        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

def get_weak_labels(candidates, u_min, manual_pk_labels=None):

    manual_pk_labels = manual_pk_labels or []
    manual_set = set((t, tuple(cols)) for t, cols in manual_pk_labels)
    labels, mask = [], []

    for c in candidates:

        if (c['table'], tuple(c['cols'])) in manual_set:

            labels.append(1)

            mask.append(True)

            continue

        u = c['features'][0]

        id_score = c['features'][7]

        is_composite = c['features'][10] > 1

        if is_composite and u > u_min:

            labels.append(1)

            mask.append(True)

        elif (not is_composite) and u > u_min and id_score >= 2:

            labels.append(1)

            mask.append(True)

        elif u < 0.90:

            labels.append(0)

            mask.append(True)

        else:

            labels.append(0)

            mask.append(False)

    return torch.tensor(labels, dtype=torch.long), torch.tensor(mask, dtype=torch.bool)

def create_train_val_masks(mask, val_ratio=0.2, seed=42):

    labeled_idx = torch.where(mask)[0].cpu().numpy()

    if len(labeled_idx) < 4:

        return None, None

    rng = np.random.default_rng(seed)

    rng.shuffle(labeled_idx)

    n_val = max(1, int(len(labeled_idx) * val_ratio))

    val_idx = labeled_idx[:n_val]

    train_idx = labeled_idx[n_val:]

    if len(train_idx) == 0:

        return None, None

    train_mask = torch.zeros_like(mask)

    val_mask = torch.zeros_like(mask)

    train_mask[train_idx] = True

    val_mask[val_idx] = True

    return train_mask, val_mask

def eval_split(log_probs, labels, split_mask):

    if split_mask is None or int(split_mask.sum()) == 0:

        return {'acc': 0.0, 'prec': 0.0, 'rec': 0.0, 'f1': 0.0}

    preds = torch.argmax(log_probs[split_mask], dim=1).cpu().numpy()

    y_true = labels[split_mask].cpu().numpy()

    return {

        'acc': accuracy_score(y_true, preds),

        'prec': precision_score(y_true, preds, zero_division=0),

        'rec': recall_score(y_true, preds, zero_division=0),

        'f1': f1_score(y_true, preds, zero_division=0)

    }

def acceptance_check(train_metrics, val_metrics, cfg):

    issues = []

    if val_metrics['acc'] < cfg.get('acceptance_min_acc', 0.90):

        issues.append('low validation accuracy')

    if val_metrics['prec'] < cfg.get('acceptance_min_precision', 0.85):

        issues.append('low validation precision')

    if val_metrics['rec'] < cfg.get('acceptance_min_recall', 0.90):

        issues.append('low validation recall')

    if val_metrics['f1'] < cfg.get('acceptance_min_f1', 0.90):

        issues.append('low validation F1')

    gap = abs(train_metrics['f1'] - val_metrics['f1'])

    if gap > cfg.get('acceptance_max_train_val_f1_gap', 0.08):

        issues.append(f'train/val F1 gap too high ({gap:.3f})')

    return len(issues) == 0, issues

def train_model(model, data, labels, train_mask, val_mask, cfg):

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])

    y_train = labels[train_mask].cpu().numpy()

    pos = float((y_train == 1).sum())

    neg = float((y_train == 0).sum())

    w_pos = max(1.0, neg / max(pos, 1.0))

    class_weights = torch.tensor([1.0, w_pos], dtype=torch.float)

    criterion = nn.NLLLoss(weight=class_weights)

    best_val_f1 = -1.0

    best_state = None

    print(f"\nPhase 3: GNN Training ({cfg['epochs']} Epochs, NLLLoss)...")

    print(f"{'Epoch':<6} | {'Loss':<8} | {'Train F1':<9} | {'Val Acc':<7} | {'Val Prec':<8} | {'Val Rec':<7} | {'Val F1':<6}")

    print('-' * 90)

    for epoch in range(cfg['epochs']):

        model.train()

        optimizer.zero_grad()

        log_probs = model(data)

        loss = criterion(log_probs[train_mask], labels[train_mask])

        loss.backward()

        optimizer.step()

        if epoch % 25 == 0 or epoch == cfg['epochs'] - 1:

            model.eval()

            with torch.no_grad():

                log_probs = model(data)

                train_m = eval_split(log_probs, labels, train_mask)

                val_m = eval_split(log_probs, labels, val_mask)

            print(f"{epoch:<6} | {loss.item():.4f}   | {train_m['f1']:.2f}      | {val_m['acc']:.2f}    | {val_m['prec']:.2f}     | {val_m['rec']:.2f}    | {val_m['f1']:.2f}")

            if val_m['f1'] >= best_val_f1:

                best_val_f1 = val_m['f1']

                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

    print('-' * 90)

    if best_state is not None:

        model.load_state_dict(best_state)

def tune_hyperparams(candidates, base_cfg):

    # Lightweight weak-label tuning before full run.

    print('\nPhase 2.5: Hyperparameter scan...')

    search = [

        {'tau': 0.20, 'u_min': 0.985, 'heads': 2, 'hidden_dim': 48, 'dropout': 0.10},

        {'tau': 0.25, 'u_min': 0.990, 'heads': 4, 'hidden_dim': 64, 'dropout': 0.10},

        {'tau': 0.30, 'u_min': 0.995, 'heads': 4, 'hidden_dim': 96, 'dropout': 0.20},

    ]

    best = None

    best_score = -1.0

    original = dict(base_cfg)

    for trial in search:

        base_cfg.update(trial)

        data = build_graph_fast(candidates)

        labels, weak_mask = get_weak_labels(candidates, base_cfg['u_min'], base_cfg.get('manual_pk_labels'))

        train_mask, val_mask = create_train_val_masks(weak_mask)

        if train_mask is None:

            continue

        model = SieveGNN(in_dim=data.x.size(1), hidden_dim=base_cfg['hidden_dim'], heads=base_cfg['heads'], dropout=base_cfg['dropout'], edge_dim=data.edge_attr.size(1))

        optimizer = torch.optim.Adam(model.parameters(), lr=base_cfg['lr'], weight_decay=base_cfg['weight_decay'])

        criterion = nn.NLLLoss()

        for _ in range(30):

            model.train()

            optimizer.zero_grad()

            log_probs = model(data)

            loss = criterion(log_probs[train_mask], labels[train_mask])

            loss.backward()

            optimizer.step()

        model.eval()

        with torch.no_grad():

            val_m = eval_split(model(data), labels, val_mask)

        if val_m['f1'] > best_score:

            best_score = val_m['f1']

            best = dict(trial)

    base_cfg.update(original)

    if best is not None:

        base_cfg.update(best)

        print(f"  > Best trial: {best} (val F1={best_score:.2f})")

    else:

        print('  > Hyperparameter scan skipped (insufficient labels).')

def generate_reports(candidates, probs):

    print('Phase 4: Generating JSON & Data Quality Profile...')

    schema = {'tables': {}}

    table_groups = defaultdict(list)

    for i, c in enumerate(candidates):

        table_groups[c['table']].append((probs[i], c))

    for table, items in table_groups.items():

        valid = [x for x in items if x[1]['features'][0] > CONFIG['u_min']]

        if not valid:

            continue

        tbl_name = clean_table_name(table)

        def pk_sort_key(item, _tbl=tbl_name):

            prob, cand = item

            cols = cand['cols']

            id_score = cand['features'][7]

            self_ref = 0

            if len(cols) == 1:

                col_base = re.sub(r'_?id$', '', cols[0].lower())

                if col_base == _tbl:

                    self_ref = 1

            return (len(cols), -id_score, -self_ref, -prob)

        valid.sort(key=pk_sort_key)

        best_prob, best_cand = valid[0]

        reasons = {

            'confidence': f"{best_prob:.2f}",

            'uniqueness': f"{best_cand['features'][0]:.1%}",

            'null_rate': f"{best_cand['features'][1]:.1%}",

            'duplicate_rate': f"{best_cand['features'][2]:.1%}",

            'id_score': f"{best_cand['features'][7]:.1f}",

            'type_consistency': f"{best_cand['features'][14]:.2f}",

            'why_top': 'high uniqueness + low nulls + id-like token and strong model score'

        }

        schema['tables'][table] = [{

            'type': 'PK',

            'columns': list(best_cand['cols']),

            'explainability': reasons

        }]

    dq = []

    for c in candidates:

        if len(c['cols']) != 1:

            continue

        p = c.get('profile', {})

        dq.append({

            'Table': c['table'],

            'Column': c['cols'][0],

            'Rows': c.get('n_rows', 0),

            'Uniqueness': f"{c['features'][0]:.1%}",

            'Nulls': f"{c['features'][1]:.1%}",

            'Duplicates': f"{c['features'][2]:.1%}",

            'Cardinality': int(round(c.get('cardinality', 0))),

            'Entropy': round(c['features'][4], 4),

            '%NumericParseable': f"{100.0 * p.get('numeric_parseable', 0.0):.1f}%",

            '%DateParseable': f"{100.0 * p.get('date_parseable', 0.0):.1f}%",

            'TypeConsistency': f"{100.0 * p.get('type_consistency', 0.0):.1f}%",

            'IDLikeScore': c['features'][7],

            'UUIDNameFlag': int(c['features'][8] > 0),

            'InvalidUUIDRate': f"{100.0 * p.get('invalid_uuid_rate', 0.0):.1f}%",

            'AvgStringLength': round(p.get('avg_len', 0.0), 2),

            'TopFrequencies': p.get('top_freq', '')

        })

    pd.DataFrame(dq).to_csv('data_quality_profile.csv', index=False)

    print('  > Saved: data_quality_profile.csv')

    return schema

##### ==========================================#####

##### MAIN ORCHESTRATOR                         #####

##### ==========================================#####

def main():

    print(f"--- Sieve-GNN: Pipeline Start ({CONFIG['num_workers']} Workers) ---")

    files = glob.glob(os.path.join(CONFIG['folder'], '*.csv'))

    if not files:

        print('No CSVs found.')

        return

    with open('sieve_workers.py', 'w') as f:

        f.write(WORKER_CODE)

    spec = importlib.util.spec_from_file_location('sieve_workers', 'sieve_workers.py')

    workers = importlib.util.module_from_spec(spec)

    sys.modules['sieve_workers'] = workers

    spec.loader.exec_module(workers)

    print(f"Phase 1: Profiling {len(files)} tables...")

    all_candidates = []

    with ProcessPoolExecutor(max_workers=CONFIG['num_workers']) as executor:

        futures = [executor.submit(workers.process_table_file, f) for f in files]

        for i, fut in enumerate(as_completed(futures)):

            res = fut.result()

            if res:

                all_candidates.extend(res)

            if i % 50 == 0:

                print(f"  Processed {i}/{len(files)}...", end='\r')

    print(f"\n  > Candidates: {len(all_candidates)}")

    if len(all_candidates) < 2:

        return

    if CONFIG.get('auto_tune', False):

        tune_hyperparams(all_candidates, CONFIG)

    data = build_graph_fast(all_candidates)

    labels, weak_mask = get_weak_labels(all_candidates, CONFIG['u_min'], CONFIG.get('manual_pk_labels'))

    train_mask, val_mask = create_train_val_masks(weak_mask)

    if train_mask is None or val_mask is None:

        print('Error: Not enough weak labels to create train/val split.')

        return

    print(f"\n  > Weak labels: train={int(train_mask.sum())}, val={int(val_mask.sum())}")

    model = SieveGNN(

        in_dim=data.x.size(1),

        hidden_dim=CONFIG['hidden_dim'],

        heads=CONFIG['heads'],

        dropout=CONFIG['dropout'],

        edge_dim=data.edge_attr.size(1)

    )

    train_model(model, data, labels, train_mask, val_mask, CONFIG)

    model.eval()

    with torch.no_grad():

        log_probs = model(data)

        probs = torch.exp(log_probs)[:, 1].cpu().numpy()

        train_metrics = eval_split(log_probs, labels, train_mask)

        val_metrics = eval_split(log_probs, labels, val_mask)

        print(

            f"\n  > Final validation: "

            f"acc={val_metrics['acc']:.2f}, "

            f"precision={val_metrics['prec']:.2f}, "

            f"recall={val_metrics['rec']:.2f}, "

            f"f1={val_metrics['f1']:.2f}"

        )

    accepted, issues = acceptance_check(train_metrics, val_metrics, CONFIG)

    if accepted:

        print('  > Acceptance check: PASS')

    else:

        print(f"  > Acceptance check: REVIEW ({'; '.join(issues)})")

    initial_schema = generate_reports(all_candidates, probs)

    final_schema = refine_and_export(all_candidates, initial_schema, data, CONFIG)

    with open('schema_final.json', 'w') as f:

        json.dump(final_schema, f, indent=2)

    with open('schema.json', 'w') as f:

        json.dump(final_schema, f, indent=2)

    print('  > Saved: schema_final.json')

    print('  > Saved: schema.json')

if __name__ == '__main__':

    import multiprocessing

    multiprocessing.freeze_support()

    main()





--- Sieve-GNN: Pipeline Start (11 Workers) ---
Phase 1: Profiling 11 tables...


<>:81: SyntaxWarning: invalid escape sequence '\d'
<>:81: SyntaxWarning: invalid escape sequence '\d'
/var/folders/kv/kkbvxprn4x97bbz4_df7r68h0000gp/T/ipykernel_11421/2802273468.py:81: SyntaxWarning: invalid escape sequence '\d'
  if re.fullmatch(r'[-+]?\d+(\.0+)?', s):
/Users/shoaib.rahman/Documents/MScThesis/paper_final/chinook/sieve_workers.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors='coerce', utc=True)
/Users/shoaib.rahman/Documents/MScThesis/paper_final/chinook/sieve_workers.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors='coerce', utc=True)
/Users/shoaib.rahman/Documents/MScThesis/paper_final/chinook/sieve_workers.py:63: UserWarni

/Users/shoaib.rahman/Documents/MScThesis/paper_final/chinook/sieve_workers.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors='coerce', utc=True)
/Users/shoaib.rahman/Documents/MScThesis/paper_final/chinook/sieve_workers.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors='coerce', utc=True)
/Users/shoaib.rahman/Documents/MScThesis/paper_final/chinook/sieve_workers.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors='coerce', utc=True)
/Users/shoaib.rahman/Documents/MScThesis/pa


  > Candidates: 304
Phase 2: Building Graph...

  > Weak labels: train=200, val=49

Phase 3: GNN Training (150 Epochs, NLLLoss)...
Epoch  | Loss     | Train F1  | Val Acc | Val Prec | Val Rec | Val F1
------------------------------------------------------------------------------------------
0      | 3.8745   | 0.79      | 0.63    | 0.79     | 0.76    | 0.78
25     | 0.3540   | 0.95      | 0.86    | 0.85     | 1.00    | 0.92
50     | 0.2029   | 0.97      | 0.92    | 0.91     | 1.00    | 0.95
75     | 0.1738   | 0.97      | 0.94    | 0.93     | 1.00    | 0.96
100    | 0.1667   | 0.98      | 0.94    | 0.93     | 1.00    | 0.96
125    | 0.1255   | 0.98      | 0.94    | 0.93     | 1.00    | 0.96
149    | 0.1164   | 0.99      | 0.94    | 0.93     | 1.00    | 0.96
------------------------------------------------------------------------------------------

  > Final validation: acc=0.94, precision=0.93, recall=1.00, f1=0.96
  > Acceptance check: PASS
Phase 4: Generating JSON & Data Quality Pro

/var/folders/kv/kkbvxprn4x97bbz4_df7r68h0000gp/T/ipykernel_11421/2802273468.py:638: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors='coerce', utc=True)
/var/folders/kv/kkbvxprn4x97bbz4_df7r68h0000gp/T/ipykernel_11421/2802273468.py:638: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors='coerce', utc=True)
/var/folders/kv/kkbvxprn4x97bbz4_df7r68h0000gp/T/ipykernel_11421/2802273468.py:638: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(s, errors='coerce', utc=True)
/var/folders/kv/kkbvxprn4x97bbz4_df7r

  > Saved: erd.png
  > Saved: schema_final.json
  > Saved: schema.json


In [3]:
##### ==========================================#####
##### MULTI-SEED EVALUATION (Statistical Robustness) #####
##### ==========================================#####
"""
Runs the GNN training + evaluation N times with different random seeds.
Reports mean ± std for PK accuracy, FK precision, recall, and F1.
Requires: sql_ddl.sql in the same directory as ground truth.
"""

import copy

DDL_PATH = "sql_ddl.sql"
NUM_SEEDS = 1
SEEDS = [2024]

def parse_ddl_ground_truth(ddl_path):
    """Parse the DDL file to extract ground-truth PKs and FKs."""
    with open(ddl_path) as f:
        ddl = f.read()

    # Extract PKs: CONSTRAINT ... PRIMARY KEY (col1, col2, ...)
    pk_gt = {}
    current_table = None
    for line in ddl.split('\n'):
        # Handle both: CREATE TABLE actor  and  CREATE TABLE "language"
        m = re.match(r'CREATE TABLE\s+"?(\w+)"?', line, re.I)
        if m:
            current_table = m.group(1).lower()
        pk_match = re.search(r'PRIMARY KEY\s*\(([^)]+)\)', line, re.I)
        if pk_match and current_table:
            cols = tuple(c.strip().strip('"').lower() for c in pk_match.group(1).split(','))
            pk_gt[current_table] = cols

    # Extract FKs: FOREIGN KEY (col) REFERENCES table(col)
    fk_gt = set()
    current_table = None
    for line in ddl.split('\n'):
        m = re.match(r'CREATE TABLE\s+"?(\w+)"?', line, re.I)
        if m:
            current_table = m.group(1).lower()
        fk_match = re.search(
            r'FOREIGN KEY\s*\((\w+)\)\s*REFERENCES\s+"?(\w+)"?\s*\((\w+)\)', line, re.I
        )
        if fk_match and current_table:
            src_col = fk_match.group(1).lower()
            tgt_table = fk_match.group(2).lower()
            tgt_col = fk_match.group(3).lower()
            fk_gt.add((current_table, src_col, tgt_table, tgt_col))

    return pk_gt, fk_gt


def evaluate_against_gt(schema, relationships, pk_gt, fk_gt):
    """Compare discovered schema against ground truth."""
    # PK evaluation
    pk_correct = 0
    pk_total = len(pk_gt)
    for gt_table, gt_cols in pk_gt.items():
        for disc_table, entries in schema.get('tables', {}).items():
            disc_clean = clean_table_name(disc_table).lower()
            if disc_clean == gt_table:
                for entry in entries:
                    if entry['type'] == 'PK':
                        disc_cols = tuple(c.lower() for c in entry['columns'])
                        if disc_cols == gt_cols:
                            pk_correct += 1
                break
    pk_acc = pk_correct / pk_total if pk_total > 0 else 0

    # FK evaluation
    discovered_fks = set()
    for r in relationships:
        src_table = clean_table_name(r['Source Table']).lower()
        src_col = r['Source Col'][0].lower() if isinstance(r['Source Col'], list) else r['Source Col'].lower()
        tgt_table = clean_table_name(r['Target Table']).lower()
        tgt_col = r['Target Col'].lower()
        discovered_fks.add((src_table, src_col, tgt_table, tgt_col))

    tp = len(discovered_fks & fk_gt)
    fp = len(discovered_fks - fk_gt)
    fn = len(fk_gt - discovered_fks)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'pk_accuracy': pk_acc,
        'pk_correct': pk_correct,
        'pk_total': pk_total,
        'fk_tp': tp, 'fk_fp': fp, 'fk_fn': fn,
        'fk_precision': precision,
        'fk_recall': recall,
        'fk_f1': f1,
    }


def run_single_seed(seed, candidates_original, config):
    """Run GNN training + schema generation + FK inference for one seed."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    # Rebuild graph (edge construction is deterministic, but reset for safety)
    data = build_graph_fast(candidates_original)
    labels, mask = get_weak_labels(candidates_original, config['u_min'], config.get('manual_pk_labels'))

    if mask.sum() == 0:
        return None

    model = SieveGNN(data.x.size(1), config['hidden_dim'], config['heads'], config['dropout'], edge_dim=data.edge_attr.size(1))
    # Silence per-seed training output
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    criterion = nn.NLLLoss()
    model.train()
    for epoch in range(config['epochs']):
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out[mask], labels[mask])
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        probs = torch.exp(model(data))[:, 1].numpy()

    schema = generate_reports.__wrapped__(candidates_original, probs) if hasattr(generate_reports, '__wrapped__') else _generate_reports_quiet(candidates_original, probs)
    relationships = infer_foreign_keys(candidates_original, schema, data, config)
    return schema, relationships


def _generate_reports_quiet(candidates, probs):
    """generate_reports without print statements."""
    schema = {"tables": {}}
    table_groups = defaultdict(list)
    for i, c in enumerate(candidates):
        table_groups[c['table']].append((probs[i], c))

    for table, items in table_groups.items():
        valid = [x for x in items if x[1]['features'][0] > 0.99]
        if not valid:
            continue
        tbl_name = clean_table_name(table)
        def pk_sort_key(item, _tbl=tbl_name):
            prob, cand = item
            cols = cand['cols']
            id_score = cand['features'][7]
            self_ref = 0
            if len(cols) == 1:
                col_base = re.sub(r'_?id$', '', cols[0].lower())
                if col_base == _tbl:
                    self_ref = 1
            return (len(cols), -id_score, -self_ref, -prob)
        valid.sort(key=pk_sort_key)
        best_prob, best_cand = valid[0]
        schema["tables"][table] = [{
            "type": "PK", "columns": list(best_cand['cols']),
            "explainability": {"confidence": f"{best_prob:.2f}", "uniqueness": f"{best_cand['features'][0]:.1%}"}
        }]
    return schema


def run_multi_seed_evaluation():
    """Main entry point for multi-seed evaluation."""
    print(f"\n{'='*60}")
    print(f"  MULTI-SEED EVALUATION ({NUM_SEEDS} seeds: {SEEDS})")
    print(f"{'='*60}\n")

    # Parse ground truth
    pk_gt, fk_gt = parse_ddl_ground_truth(DDL_PATH)
    print(f"Ground truth: {len(pk_gt)} PKs, {len(fk_gt)} FKs\n")

    # Profile once (deterministic)
    files = glob.glob(os.path.join(CONFIG['folder'], "*.csv"))
    with open("sieve_workers.py", "w") as f:
        f.write(WORKER_CODE)
    spec = importlib.util.spec_from_file_location("sieve_workers", "sieve_workers.py")
    workers = importlib.util.module_from_spec(spec)
    sys.modules["sieve_workers"] = workers
    spec.loader.exec_module(workers)

    print("Profiling (once, shared across seeds)...")
    all_candidates = []
    with ProcessPoolExecutor(max_workers=CONFIG['num_workers']) as executor:
        futures = [executor.submit(workers.process_table_file, f) for f in files]
        for fut in as_completed(futures):
            res = fut.result()
            if res:
                all_candidates.extend(res)
    print(f"  > {len(all_candidates)} candidates\n")

    # Run each seed
    all_results = []
    for seed in SEEDS:
        print(f"Seed {seed}...", end=" ")
        result = run_single_seed(seed, all_candidates, CONFIG)
        if result is None:
            print("FAILED (no labels)")
            continue
        schema, relationships = result
        metrics = evaluate_against_gt(schema, relationships, pk_gt, fk_gt)
        all_results.append(metrics)
        print(f"PK={metrics['pk_correct']}/{metrics['pk_total']}  "
              f"FK P={metrics['fk_precision']:.3f} R={metrics['fk_recall']:.3f} F1={metrics['fk_f1']:.3f}")

    # Summary statistics
    print(f"\n{'='*60}")
    print("  SUMMARY (mean ± std)")
    print(f"{'='*60}")
    for metric_name in ['pk_accuracy', 'fk_precision', 'fk_recall', 'fk_f1']:
        values = [r[metric_name] for r in all_results]
        mean_v = np.mean(values)
        std_v = np.std(values)
        print(f"  {metric_name:<15}: {mean_v:.4f} ± {std_v:.4f}")

    return all_results

# Run it
multi_seed_results = run_multi_seed_evaluation()


  MULTI-SEED EVALUATION (5 seeds: [42, 123, 256, 789, 1024])

Ground truth: 11 PKs, 11 FKs

Profiling (once, shared across seeds)...
  > 480 candidates

Seed 42... Phase 2: Building Graph...

  Phase 5a: Inferring Foreign Keys...
    > FK weak labels: pos=9, neg=1, total=10
    > FK edge model trained (train=8, val=2, best val F1=1.000)
    > 10 FK candidates from strategy cascade
    > 9 FK relationships validated with data
PK=11/11  FK P=1.000 R=0.818 F1=0.900
Seed 123... Phase 2: Building Graph...

  Phase 5a: Inferring Foreign Keys...
    > FK weak labels: pos=9, neg=1, total=10
    > FK edge model trained (train=8, val=2, best val F1=1.000)
    > 10 FK candidates from strategy cascade
    > 9 FK relationships validated with data
PK=11/11  FK P=1.000 R=0.818 F1=0.900
Seed 256... Phase 2: Building Graph...

  Phase 5a: Inferring Foreign Keys...
    > FK weak labels: pos=9, neg=1, total=10
    > FK edge model trained (train=8, val=2, best val F1=1.000)
    > 10 FK candidates from st

In [6]:
##### ==========================================#####
##### HEURISTIC-ONLY ABLATION (No GNN)          #####
##### ==========================================#####
"""
Runs the full pipeline but with GNN disabled:
  - All candidates get probability = 0.0 (GNN contributes nothing)
  - PK selection uses only: minimality, ID score, self-reference
  - FK inference uses only: name similarity + data inclusion (no graph edges)
Compares against full Sieve-GNN to quantify the GNN's contribution.
"""


def run_heuristic_only_ablation():
    """Run pipeline without GNN and compare to full pipeline."""
    print(f"\n{'='*60}")
    print("  ABLATION: HEURISTIC-ONLY (no GNN)")
    print(f"{'='*60}\n")

    pk_gt, fk_gt = parse_ddl_ground_truth(DDL_PATH)
    print(f"Ground truth: {len(pk_gt)} PKs, {len(fk_gt)} FKs\n")

    # Profile (reuse if already done, otherwise redo)
    files = glob.glob(os.path.join(CONFIG['folder'], "*.csv"))
    with open("sieve_workers.py", "w") as f:
        f.write(WORKER_CODE)
    spec = importlib.util.spec_from_file_location("sieve_workers", "sieve_workers.py")
    workers = importlib.util.module_from_spec(spec)
    sys.modules["sieve_workers"] = workers
    spec.loader.exec_module(workers)

    print("Profiling...")
    all_candidates = []
    with ProcessPoolExecutor(max_workers=CONFIG['num_workers']) as executor:
        futures = [executor.submit(workers.process_table_file, f) for f in files]
        for fut in as_completed(futures):
            res = fut.result()
            if res:
                all_candidates.extend(res)
    print(f"  > {len(all_candidates)} candidates\n")

    # ---- FULL PIPELINE (with GNN) ----
    print("Running FULL pipeline (with GNN)...")
    torch.manual_seed(42)
    np.random.seed(42)
    data_full = build_graph_fast(all_candidates)
    labels, mask = get_weak_labels(all_candidates, CONFIG['u_min'], CONFIG.get('manual_pk_labels'))

    model = SieveGNN(data.x.size(1), CONFIG['hidden_dim'], CONFIG['heads'], CONFIG['dropout'], edge_dim=data.edge_attr.size(1))
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    criterion = nn.NLLLoss()
    model.train()
    for epoch in range(CONFIG['epochs']):
        optimizer.zero_grad()
        out = model(data_full)
        loss = criterion(out[mask], labels[mask])
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        probs_full = torch.sigmoid(model(data_full)).squeeze().numpy()

    schema_full = _generate_reports_quiet(all_candidates, probs_full)
    rels_full = infer_foreign_keys(all_candidates, schema_full, data_full, CONFIG)
    metrics_full = evaluate_against_gt(schema_full, rels_full, pk_gt, fk_gt)

    # ---- HEURISTIC-ONLY (GNN disabled) ----
    print("\nRunning HEURISTIC-ONLY pipeline (GNN probability = 0 for all)...")
    probs_zero = np.zeros(len(all_candidates))  # All GNN probs = 0

    schema_heur = _generate_reports_quiet(all_candidates, probs_zero)

    # For FK: only use Strategy 1 (name similarity + data inclusion), skip graph edges
    # We build a dummy graph with no edges to disable Strategy 2
    dummy_edge_index = torch.tensor([[0], [0]], dtype=torch.long)
    dummy_edge_attr = torch.zeros((1, 4), dtype=torch.float)
    x = torch.tensor([c['features'] for c in all_candidates], dtype=torch.float).nan_to_num()
    data_dummy = Data(x=x, edge_index=dummy_edge_index, edge_attr=dummy_edge_attr)

    rels_heur = infer_foreign_keys(all_candidates, schema_heur, data_dummy, CONFIG)
    metrics_heur = evaluate_against_gt(schema_heur, rels_heur, pk_gt, fk_gt)

    # ---- COMPARISON TABLE ----
    print(f"\n{'='*60}")
    print("  ABLATION RESULTS")
    print(f"{'='*60}")
    print(f"{'Metric':<20} {'Heuristic-Only':>15} {'Full Sieve-GNN':>15} {'Delta':>10}")
    print("-" * 62)

    comparisons = [
        ('PK Accuracy', 'pk_accuracy'),
        ('FK Precision', 'fk_precision'),
        ('FK Recall', 'fk_recall'),
        ('FK F1', 'fk_f1'),
    ]
    for label, key in comparisons:
        h = metrics_heur[key]
        f = metrics_full[key]
        delta = f - h
        sign = "+" if delta >= 0 else ""
        print(f"  {label:<18} {h:>14.4f} {f:>14.4f} {sign}{delta:>9.4f}")

    print(f"\n  PK detail:  Heuristic={metrics_heur['pk_correct']}/{metrics_heur['pk_total']}  "
          f"Full={metrics_full['pk_correct']}/{metrics_full['pk_total']}")
    print(f"  FK detail:  Heuristic TP={metrics_heur['fk_tp']} FP={metrics_heur['fk_fp']} FN={metrics_heur['fk_fn']}  "
          f"Full TP={metrics_full['fk_tp']} FP={metrics_full['fk_fp']} FN={metrics_full['fk_fn']}")

    # Per-table PK comparison to find where GNN matters
    print(f"\n  Per-table PK differences:")
    pk_diffs_found = False
    for gt_table, gt_cols in pk_gt.items():
        pk_heur = None
        pk_full = None
        for disc_table, entries in schema_heur.get('tables', {}).items():
            if clean_table_name(disc_table).lower() == gt_table:
                for e in entries:
                    if e['type'] == 'PK':
                        pk_heur = tuple(e['columns'])
                break
        for disc_table, entries in schema_full.get('tables', {}).items():
            if clean_table_name(disc_table).lower() == gt_table:
                for e in entries:
                    if e['type'] == 'PK':
                        pk_full = tuple(e['columns'])
                break
        if pk_heur != pk_full:
            pk_diffs_found = True
            print(f"    {gt_table}: Heuristic={pk_heur} vs Full={pk_full} (GT={gt_cols})")

    if not pk_diffs_found:
        print("    No differences - GNN tiebreaker was not needed for any table.")

    return metrics_heur, metrics_full

# Run it
ablation_heur, ablation_full = run_heuristic_only_ablation()


  ABLATION: HEURISTIC-ONLY (no GNN)

Ground truth: 15 PKs, 18 FKs

Profiling...
  > 414 candidates

Running FULL pipeline (with GNN)...
Phase 2: Building Graph...

  Phase 5a: Inferring Foreign Keys...
    > 23 FK candidates from name + graph signals
    > 20 FK relationships validated with data

Running HEURISTIC-ONLY pipeline (GNN probability = 0 for all)...

  Phase 5a: Inferring Foreign Keys...
    > 23 FK candidates from name + graph signals
    > 20 FK relationships validated with data

  ABLATION RESULTS
Metric                Heuristic-Only  Full Sieve-GNN      Delta
--------------------------------------------------------------
  PK Accuracy                0.9333         0.9333 +   0.0000
  FK Precision               0.8500         0.8500 +   0.0000
  FK Recall                  0.9444         0.9444 +   0.0000
  FK F1                      0.8947         0.8947 +   0.0000

  PK detail:  Heuristic=14/15  Full=14/15
  FK detail:  Heuristic TP=17 FP=3 FN=1  Full TP=17 FP=3 FN=1

 